In [1]:
import os
import numpy as np
import tensorflow as tf

from common import calculate_final_metrics, calculate_f1

MODEL_PATH = "/home/dani/Documents/tugas akhir/TugasAkhirku2026/mrtcn_ids/Model/FL/imgsize256D055/saved_models/model_final_noniid.h5"

BASE_PATH = "/home/dani/Documents/tugas akhir/TugasAkhirku2026/mrtcn_ids/Preprocessing/hasil/final/preprocessingimgsize256_s16/"
TEST_PATH = os.path.join(BASE_PATH, "forFL/test_global.npz")

2026-04-27 13:14:38.029705: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-27 13:14:38.041480: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777270478.056535  390080 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777270478.061337  390080 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777270478.072633  390080 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
model = tf.keras.models.load_model(MODEL_PATH)

print("Model loaded")
model.summary()

I0000 00:00:1777270480.407385  390080 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4144 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


Model loaded


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 256, 64)        │        49,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 256, 64)        │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 256, 64)        │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 32, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         2,049 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 75,971 (296.77 KB)

 Trainable params: 75,969 (296.75 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [3]:
import numpy as np

data = np.load(TEST_PATH, mmap_mode="r")

x_test = data["x_test"]
y_test = data["y_test"]

print("Shape:", x_test.shape)
print("Dtype:", x_test.dtype)

Shape: (12365, 256, 256)
Dtype: float32


In [4]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import confusion_matrix

# ==============================
# 1. PASTIKAN DATA VALID
# ==============================
x_test = x_test.astype("float32")
y_test = y_test.astype("float32")

In [5]:
# ==============================
# 2. DATASET (UNTUK INFERENCE)
# ==============================
batch_size = 16
test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test)).batch(batch_size)

In [6]:
# ==============================
# 3. PREDICT (WAJIB UNTUK METRIC FINAL)
# ==============================
y_pred_prob = model.predict(test_ds, verbose=1)
# threshold default
y_pred = (y_pred_prob > 0.5).astype(int)

I0000 00:00:1777270491.253757  390261 service.cc:152] XLA service 0x715690004bc0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777270491.253775  390261 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 3060 Laptop GPU, Compute Capability 8.6
2026-04-27 13:14:51.261623: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1777270491.285133  390261 cuda_dnn.cc:529] Loaded cuDNN version 91701


 77/773 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step

I0000 00:00:1777270491.718442  390261 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


773/773 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step


In [7]:
# ==============================
# 4. AMBIL y_true (GLOBAL)
# ==============================
y_true = np.concatenate([y.numpy() for _, y in test_ds], axis=0)

2026-04-27 13:14:54.052993: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [8]:
# ==============================
# 5. NORMALISASI SHAPE & LABEL
# ==============================
y_true = (y_true.reshape(-1) > 0.5).astype(int)
y_pred = y_pred.reshape(-1)


In [9]:
# ==============================
# 6. CONFUSION MATRIX (FIX ORDER)
# ==============================
tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()


In [10]:
# ==============================
# 7. METRICS (GLOBAL, VALID)
# ==============================
accuracy  = (tp + tn) / (tp + tn + fp + fn)

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

# ==============================
# 8. OUTPUT (FORMAT LAPORAN)
# ==============================
print("\n===== HASIL EVALUASI MODEL =====")
print(f"Accuracy  : {accuracy:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-Score  : {f1:.6f}")
print(f"FPR       : {fpr:.6f}")
print(f"FNR       : {fnr:.6f}")

print("\nConfusion Matrix:")
print(f"TN: {tn} | FP: {fp}")
print(f"FN: {fn} | TP: {tp}")


===== HASIL EVALUASI MODEL =====
Accuracy  : 0.577922
Precision : 0.535093
Recall    : 0.957949
F1-Score  : 0.686641
FPR       : 0.776735
FNR       : 0.042051

Confusion Matrix:
TN: 1428 | FP: 4968
FN: 251 | TP: 5718


In [1]:
import os
import numpy as np
import tensorflow as tf

from common import calculate_final_metrics, calculate_f1

MODEL_PATH = "/home/dani/Documents/tugas akhir/TugasAkhirku2026/mrtcn_ids/Model/FL/imgsize256D055/saved_models/model_best_noniid.h5"

BASE_PATH = "/home/dani/Documents/tugas akhir/TugasAkhirku2026/mrtcn_ids/Preprocessing/hasil/final/preprocessingimgsize256_s16/"
TEST_PATH = os.path.join(BASE_PATH, "forFL/test_global.npz")

2026-04-27 13:15:17.876447: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-27 13:15:17.887566: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777270517.900651  391161 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777270517.904838  391161 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777270517.915148  391161 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
model = tf.keras.models.load_model(MODEL_PATH)

print("Model loaded")
model.summary()

I0000 00:00:1777270520.042699  391161 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4144 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


Model loaded


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 256, 64)        │        49,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 256, 64)        │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 256, 64)        │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 32, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         2,049 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 75,971 (296.77 KB)

 Trainable params: 75,969 (296.75 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [3]:
import numpy as np

data = np.load(TEST_PATH, mmap_mode="r")

x_test = data["x_test"]
y_test = data["y_test"]

print("Shape:", x_test.shape)
print("Dtype:", x_test.dtype)

Shape: (12365, 256, 256)
Dtype: float32


In [4]:
# ==============================
# 1. PASTIKAN DATA VALID
# ==============================
x_test = x_test.astype("float32")
y_test = y_test.astype("float32")

In [5]:
# ==============================
# 2. DATASET (UNTUK INFERENCE)
# ==============================
batch_size = 16
test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test)).batch(batch_size)

In [6]:
# ==============================
# 3. PREDICT (WAJIB UNTUK METRIC FINAL)
# ==============================
y_pred_prob = model.predict(test_ds, verbose=1)
# threshold default
y_pred = (y_pred_prob > 0.5).astype(int)

I0000 00:00:1777270530.934916  391373 service.cc:152] XLA service 0x733864005ee0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777270530.934934  391373 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 3060 Laptop GPU, Compute Capability 8.6
2026-04-27 13:15:30.942550: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1777270530.965518  391373 cuda_dnn.cc:529] Loaded cuDNN version 91701


 72/773 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step

I0000 00:00:1777270531.410096  391373 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


773/773 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step


In [7]:
# ==============================
# 4. AMBIL y_true (GLOBAL)
# ==============================
y_true = np.concatenate([y.numpy() for _, y in test_ds], axis=0)

2026-04-27 13:15:33.837912: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [8]:
# ==============================
# 5. NORMALISASI SHAPE & LABEL
# ==============================
y_true = (y_true.reshape(-1) > 0.5).astype(int)
y_pred = y_pred.reshape(-1)


In [9]:
# ==============================
# 6. CONFUSION MATRIX (FIX ORDER)
# ==============================
from sklearn.metrics import confusion_matrix
tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()


In [10]:
# ==============================
# 7. METRICS (GLOBAL, VALID)
# ==============================
accuracy  = (tp + tn) / (tp + tn + fp + fn)

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

# ==============================
# 8. OUTPUT (FORMAT LAPORAN)
# ==============================
print("\n===== HASIL EVALUASI MODEL =====")
print(f"Accuracy  : {accuracy:.6f}")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1-Score  : {f1:.6f}")
print(f"FPR       : {fpr:.6f}")
print(f"FNR       : {fnr:.6f}")

print("\nConfusion Matrix:")
print(f"TN: {tn} | FP: {fp}")
print(f"FN: {fn} | TP: {tp}")


===== HASIL EVALUASI MODEL =====
Accuracy  : 0.577922
Precision : 0.535093
Recall    : 0.957949
F1-Score  : 0.686641
FPR       : 0.776735
FNR       : 0.042051

Confusion Matrix:
TN: 1428 | FP: 4968
FN: 251 | TP: 5718
